In [ ]:
import torch
from pathlib import Path

# 1. 載入專案既存的資料處理與視覺轉換模組
from src.data import build_default_vocabs, get_encoder_spec, collate_fn
from src.data.vocabulary import build_joint_ctc_vocab, Vocabulary
from src.data.dataset import PreRenderedOMRDataset

# 2. 載入新實作的 CRNN 與 CTC 損失、解碼核心
from src.model.encoders import ResNetEncoder
from src.model.crnn import OMRCRNNModel
from src.model.ctc_losses import compute_ctc_loss, pack_streams_to_ctc_targets
from src.postproc.ctc_decode import ctc_greedy_decode_batch
from src.postproc.decode import ids_to_tuples
from src.postproc.jianpu import tuples_to_jianpu, JianpuRenderConfig

print("===== Step 1: 全域詞表與一體化聯合字典建置 =====")
vb = build_default_vocabs()
joint_token_to_id, id_to_joint_token = build_joint_ctc_vocab(vb)
print(f" 聯合詞表組裝成功！CTC 總類別數: {len(id_to_joint_token)}")

print("\n===== Step 2: 嘗試載入驗證集內的真實樂譜影像與標籤 =====")
val_manifest = Path("data/synthetic/val/manifest.jsonl")
spec = get_encoder_spec("resnet")
eval_transform = spec.build_eval_transform()

if val_manifest.exists():
    print(f" 偵測到真實驗證集，正在讀取: {val_manifest}")
    # 實實例化真實 Dataset
    dataset = PreRenderedOMRDataset(
        manifest_path=val_manifest, vocabs=vb, transform=eval_transform, max_seq_len=512
    )
    # 抽出第一個真實樣本進行端到端解碼測試
    raw_item = dataset[0]
    # 利用專案現有的 collate_fn 自動完成變長影像與遮罩的 Batch 打包契約
    batch = collate_fn([raw_item])
    is_mock = False
    print(f" 成功載入真實樣本！影像張量形狀: {batch['pixel_values'].shape}")
else:
    print(" 未偵測到實體驗證集，自動切換至『虛擬模擬防禦模式』...")
    # 構造一個 Batch=1, Channel=1, H=128, W=400 的模擬 Tensor
    # 模擬背景為 1.0 (白紙)，並畫上幾條接近 0.0 的黑色五線譜橫線特徵以測試遮罩
    mock_pixels = torch.ones(1, 1, 128, 400)
    mock_pixels[:, :, [40, 50, 60, 70, 80], 20:300] = 0.0  # 畫上模擬譜線與音符

    # 虛擬一組 Lock-step 樂譜標籤，模擬經過 DataLoader 的 Padding 狀態
    mock_batch = {
        "pixel_values": mock_pixels,
        "label_lengths": torch.tensor([5], dtype=torch.long),
        "type_ids": torch.tensor(
            [
                [
                    vb.type.token_to_id["clef"],
                    vb.type.token_to_id["time_signature"],
                    vb.type.token_to_id["note"],
                    Vocabulary.EOS_ID,
                    0,
                ]
            ],
            dtype=torch.long,
        ),
        "pitch_ids": torch.tensor(
            [
                [
                    Vocabulary.NULL_ID,
                    Vocabulary.NULL_ID,
                    vb.pitch.token_to_id["C4"],
                    Vocabulary.EOS_ID,
                    0,
                ]
            ],
            dtype=torch.long,
        ),
        "rhythm_ids": torch.tensor(
            [
                [
                    Vocabulary.NULL_ID,
                    Vocabulary.NULL_ID,
                    vb.rhythm.token_to_id["quarter"],
                    Vocabulary.EOS_ID,
                    0,
                ]
            ],
            dtype=torch.long,
        ),
        "attribute_ids": torch.tensor(
            [
                [
                    vb.attribute.token_to_id["G2"],
                    vb.attribute.token_to_id["4/4"],
                    Vocabulary.NULL_ID,
                    Vocabulary.EOS_ID,
                    0,
                ]
            ],
            dtype=torch.long,
        ),
    }
    batch = mock_batch
    is_mock = True
    print(f" 虛擬模擬 Batch 建立完畢。影像形狀: {batch['pixel_values'].shape}")

print("\n===== Step 3: 驗證多流標籤一體化打包 (Pack Streams) =====")
ctc_targets, target_lengths = pack_streams_to_ctc_targets(batch, vb, joint_token_to_id)
print(f" CTC 聯合 Target 矩陣形狀: {ctc_targets.shape}")
print(f" 各樣本實際目標 Token 長度: {target_lengths.tolist()}")

print("\n===== Step 4: 視覺模型前向傳播與高度稀疏遮罩校驗 =====")
encoder = ResNetEncoder(d_model=384, encoder_spec=spec)
model = OMRCRNNModel(
    encoder=encoder, ctc_vocab_size=len(id_to_joint_token), d_model=384
)

# 執行前向傳播
model_out = model(batch["pixel_values"])
logits = model_out["logits"]
attention_mask = model_out["attention_mask"]

# 計算遮罩釋放的時間步長
input_lengths = attention_mask.sum(dim=-1)
print(f" 影像壓扁後的總時間步長 (T): {logits.shape[1]}")
print(f" 經過我們修正後的遮罩有效步長 (input_lengths): {input_lengths.tolist()}")

# 關鍵防禦工程斷言：有效步長絕對不能為 0
assert (
    input_lengths.min() > 0
), " 錯誤！偵測到遮罩有效長度為 0，請確認是否已正確修改 encoders.py 中的閾值！"
print(" 遮罩閾值檢查完全通過！五線譜稀疏特徵已成功放行。")

print("\n===== Step 5: 驗證 CTC 損失函數與反向傳播 (Loss & Backward) =====")
loss = compute_ctc_loss(
    logits=logits,
    ctc_targets=ctc_targets,
    attention_mask=attention_mask,
    target_lengths=target_lengths,
    blank_id=0,
)
print(f" 成功算出當前 Batch 的 CTC Loss 標量值: {loss.item():.4f}")
loss.backward()
print(" 成功執行 loss.backward()！運算圖完整，梯度可以正常回傳更新權重。")

print(
    "\n===== Step 6: 終極測試：真實樂譜大比對 (Ground Truth vs Model Prediction) ====="
)
# 使用模型內建的貪婪解碼器預測 Token IDs
preds, lengths = model.predict_greedy(batch["pixel_values"], blank_id=0)
# 通過 ctc_greedy_decode_batch 進行去重與 Blank 過濾，還原成標準 4-Tuple 流
pred_batch_tuples = ctc_greedy_decode_batch(
    preds, lengths, id_to_joint_token, blank_id=0
)

# 提取第一個樣本的預測
sample_pred_tuples = pred_batch_tuples[0]

# 還原 Ground Truth 樣本的真實 4-Tuple 符號流
L_gt = int(batch["label_lengths"][0].item())
sample_gt_tuples = ids_to_tuples(
    batch["type_ids"][0, :L_gt],
    batch["pitch_ids"][0, :L_gt],
    batch["rhythm_ids"][0, :L_gt],
    batch["attribute_ids"][0, :L_gt],
    vb,
    strict=False,
)

# 呼叫簡譜渲染組態
jcfg = JianpuRenderConfig(emit_header=True)

print("-" * 60)
if not is_mock:
    print(
        " [真實樣本對照結果] (註：因模型尚未完全收斂，目前預測噴出 UNK 或錯亂屬正常現象)"
    )
    print(
        f" 真實 Ground Truth 簡譜文字渲染結果:\n{tuples_to_jianpu(sample_gt_tuples, jcfg)}"
    )
else:
    print(" [模擬樣本對照結果]")
    print(
        f" 模擬 Ground Truth 簡譜文字渲染結果:\n{tuples_to_jianpu(sample_gt_tuples, jcfg)}"
    )

print(
    f"\n 模型當前預測預測出的簡譜文字結果:\n{tuples_to_jianpu(sample_pred_tuples, jcfg)}"
)
print("-" * 60)
print("\n 進階全管線與簡譜文字對齊測試圓滿通關！解碼與後處理模組 100% 準備就緒！")